# Limpieza de los datos
En este notebook se realizara la limpieza siguiendo el análisis obtenido en el archivo eda_data.ipynb. Durante el análisis se ideintificaron las siguiente anomalías:
- Registros duplicados
- Transacciones sin identificación
- Cantidades y precios unitarios negativos
- Outliers en cantidades y precios unitarios

## 0. Imports y Carga del Dataset

Preparamos el entorno de trabajo: importamos librerías, configuramos la estética visual, definimos las rutas del proyecto y cargamos el dataset original. A continuación creamos las columnas auxiliares necesarias para los pasos de limpieza (`Fecha`, `Mes`, `DiaSemana`, `EsCancelacion`, `TotalPrice`) e inicializamos el diccionario de auditoría `stats_cleaning` que registrará paso a paso cuántos registros eliminamos y por qué.


In [12]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Configuración visual global
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
%matplotlib inline

# Rutas del proyecto
RUTA_CSV      = '../../../data/raw/data.csv'
RUTA_GRAFICOS = '../../../graphics/'
RUTA_INTERIM  = '../../../data/interim/'
os.makedirs(RUTA_GRAFICOS, exist_ok=True)
os.makedirs(RUTA_INTERIM,  exist_ok=True)

print('Librerías cargadas correctamente.')


Librerías cargadas correctamente.


In [13]:
# Carga del dataset original
df_raw     = pd.read_csv(RUTA_CSV, encoding='latin-1')
df_working = df_raw.copy()

# ── Columnas auxiliares ────────────────────────────────────────────────────────
df_working['InvoiceDate']   = pd.to_datetime(df_working['InvoiceDate'], format='mixed')
df_working['Fecha']         = df_working['InvoiceDate'].dt.normalize()
df_working['Mes']           = df_working['InvoiceDate'].dt.to_period('M')
df_working['DiaSemana']     = df_working['InvoiceDate'].dt.day_name()
df_working['EsCancelacion'] = df_working['InvoiceNo'].str.startswith('C')
df_working['TotalPrice']    = df_working['Quantity'] * df_working['UnitPrice']

# ── Diccionario de auditoría (se actualiza en cada paso) ──────────────────────
stats_cleaning = {'Registros Iniciales': len(df_raw)}

# ── Resumen de carga ──────────────────────────────────────────────────────────
print('=' * 55)
print(f'  DATASET CARGADO')
print('=' * 55)
print(f'  Filas    : {df_raw.shape[0]:>10,}')
print(f'  Columnas : {df_raw.shape[1]:>10}')
print('=' * 55)
print(f'\n  df_working activo : {len(df_working):,} filas')
print(f'  Columnas auxiliares añadidas:')
print(f'    - Fecha, Mes, DiaSemana, EsCancelacion, TotalPrice')


  DATASET CARGADO
  Filas    :    541,909
  Columnas :          8

  df_working activo : 541,909 filas
  Columnas auxiliares añadidas:
    - Fecha, Mes, DiaSemana, EsCancelacion, TotalPrice


# 3. LIMPIEZA DE DATOS

### 3.1 ELIMINAR FILAS CON Description NULA

Motivo: el 100% de estas filas cumplen simultáneamente:
 - Description = NaN  → no sabemos qué producto es
 - UnitPrice = 0      → no generan ningún ingreso (TotalPrice = 0)
 - CustomerID = NaN   → no tienen cliente asociado
No son recuperables y solo añadirían ruido al modelo.

In [14]:
# 3.1 — Eliminar filas con Description nula
antes = len(df_working)
df_working = df_working.dropna(subset=['Description']).reset_index(drop=True)
eliminadas_desc = antes - len(df_working)
stats_cleaning['Description nulas eliminadas'] = eliminadas_desc

print(f"── 3.1 Eliminar filas con Description nula ──")
print(f"  Justificación: el 100% tienen UnitPrice=0 y CustomerID=NaN simultáneamente")
print(f"  → no generan ingreso ni tienen cliente → ruido puro para el modelo")
print()
print(f"  Filas antes     : {antes:,}")
print(f"  Filas eliminadas: {eliminadas_desc:,}")
print(f"  Filas después   : {len(df_working):,}")
print(f"  Verificación    — Description nulos restantes: {df_working['Description'].isnull().sum()}")


── 3.1 Eliminar filas con Description nula ──
  Justificación: el 100% tienen UnitPrice=0 y CustomerID=NaN simultáneamente
  → no generan ingreso ni tienen cliente → ruido puro para el modelo

  Filas antes     : 541,909
  Filas eliminadas: 1,454
  Filas después   : 540,455
  Verificación    — Description nulos restantes: 0


### 3.2 Eliminar duplicados exactos

Una fila duplicada exacta tiene **todos** sus campos idénticos: mismo `InvoiceNo`, `StockCode`, `Description`, `Quantity`, `UnitPrice`, `InvoiceDate`, `CustomerID` y `Country`. Esto es físicamente imposible en un sistema transaccional real — su presencia indica errores de doble inserción en la base de datos, exports corruptos o fallos de ETL. Se conserva la primera ocurrencia (`keep='first'`) al ser todas idénticas.


In [15]:
antes = len(df_working)
df_working = df_working.drop_duplicates(keep='first', ignore_index=True)
eliminadas_dup = antes - len(df_working)
stats_cleaning['Duplicados eliminados'] = eliminadas_dup

print(f"── 3.2 Eliminar duplicados exactos ──")
print(f"  Filas antes     : {antes:,}")
print(f"  Filas eliminadas: {eliminadas_dup:,}")
print(f"  Filas después   : {len(df_working):,}")
print(f"  Verificación    — duplicados restantes: {df_working.duplicated().sum()}")


── 3.2 Eliminar duplicados exactos ──
  Filas antes     : 540,455
  Filas eliminadas: 5,268
  Filas después   : 535,187
  Verificación    — duplicados restantes: 0


### 3.3 Eliminar negativos huérfanos

Son filas con `Quantity < 0` pero **sin prefijo `C`** en `InvoiceNo`. En este dataset son ajustes internos de almacén (descripciones como "faulty", "damages", "check", "reverse adjustment"). El análisis del CSV confirma que el **100%** cumple simultáneamente:
- `UnitPrice = 0.0` → `TotalPrice = 0` siempre, sin impacto en ingresos
- `CustomerID = NaN` → ninguna tiene cliente asociado
- Sin prefijo `C` → el sistema nunca las registró como cancelación oficial

No son cancelaciones de venta ni transacciones recuperables: eliminarlas es lo correcto.


In [16]:
# 3.3 — Eliminar negativos huérfanos (ajustes internos de almacén)
antes = len(df_working)
mask_huerfanos = (
    ~df_working['InvoiceNo'].str.startswith('C', na=False) &
    (df_working['Quantity'] < 0) &
    (df_working['UnitPrice'] == 0.0)
)
df_working = df_working[~mask_huerfanos].reset_index(drop=True)
eliminadas_huerfanos = antes - len(df_working)
stats_cleaning['Huérfanos eliminados'] = eliminadas_huerfanos

print(f"── 3.3 Eliminar negativos huérfanos ──")
print(f"  Justificación: sin C + Qty<0 + Price=0 → TotalPrice=0, sin cliente → ruido")
print()
print(f"  Filas antes     : {antes:,}")
print(f"  Filas eliminadas: {eliminadas_huerfanos:,}")
print(f"  Filas después   : {len(df_working):,}")
print(f"  Verificación    — huérfanos restantes: {(~df_working['InvoiceNo'].str.startswith('C', na=False) & (df_working['Quantity'] < 0) & (df_working['UnitPrice'] == 0.0)).sum()}")


── 3.3 Eliminar negativos huérfanos ──
  Justificación: sin C + Qty<0 + Price=0 → TotalPrice=0, sin cliente → ruido

  Filas antes     : 535,187
  Filas eliminadas: 474
  Filas después   : 534,713
  Verificación    — huérfanos restantes: 0


### 3.4 Eliminar StockCodes no estándar

Los códigos de producto comerciales siguen el patrón `DDDDD[LL]` (5 dígitos + hasta 2 letras opcionales). Códigos como `POST`, `DOT`, `D`, `M`, `BANK CHARGES`, `AMAZONFEE` son códigos operacionales (portes, descuentos, comisiones, ajustes manuales) que no representan ventas de producto.

In [17]:
# 3.4 — Eliminar StockCodes no estándar
# Patrón: 5 dígitos + hasta 2 letras opcionales (cubre variantes como 15056BL)
PATRON_STOCK = r'^[0-9]{5}[A-Za-z]{0,2}$'

antes = len(df_working)
mask_estandar = df_working['StockCode'].str.match(PATRON_STOCK, na=False)
df_working = df_working[mask_estandar].reset_index(drop=True)
eliminadas_stock = antes - len(df_working)
stats_cleaning['StockCodes no estándar eliminados'] = eliminadas_stock

print(f"── 3.4 Eliminar StockCodes no estándar ──")
print(f"  Patrón aplicado : {PATRON_STOCK}")
print()
print(f"  Filas antes     : {antes:,}")
print(f"  Filas eliminadas: {eliminadas_stock:,}")
print(f"  Filas después   : {len(df_working):,}")
print(f"  Verificación    — no estándar restantes: {(~df_working['StockCode'].str.match(PATRON_STOCK, na=False)).sum()}")


── 3.4 Eliminar StockCodes no estándar ──
  Patrón aplicado : ^[0-9]{5}[A-Za-z]{0,2}$

  Filas antes     : 534,713
  Filas eliminadas: 2,969
  Filas después   : 531,744
  Verificación    — no estándar restantes: 0


### 3.5 Capping (winsorización) de outliers en Quantity y UnitPrice

En lugar de eliminar filas con valores extremos, **recortamos** los valores al percentil 99 de los registros positivos (`clip`). Esto conserva todas las filas y evita perder días de ventas en la serie temporal.

El p99 se calcula **solo sobre valores positivos** para no contaminar el umbral con las cancelaciones (Quantity negativa). Tras el capping se recalcula `TotalPrice`.

> **Por qué no eliminar filas**: en una serie temporal, eliminar transacciones de un día puede hacer que ese día tenga menos ventas de las reales, introduciendo un sesgo sistemático en la variable objetivo.


In [18]:
# 3.5 — Capping (winsorización) al percentil 99 — solo sobre valores positivos
cap_qty   = df_working.loc[df_working['Quantity']  > 0, 'Quantity'].quantile(0.99)
cap_price = df_working.loc[df_working['UnitPrice'] > 0, 'UnitPrice'].quantile(0.99)

n_qty_sup   = (df_working['Quantity']  >  cap_qty).sum()
n_qty_inf   = (df_working['Quantity']  < -cap_qty).sum()
n_price_sup = (df_working['UnitPrice'] >  cap_price).sum()

df_working['Quantity']  = df_working['Quantity'].clip(lower=-cap_qty, upper=cap_qty)
df_working['UnitPrice'] = df_working['UnitPrice'].clip(upper=cap_price)
df_working['TotalPrice'] = df_working['Quantity'] * df_working['UnitPrice']

stats_cleaning['Filas capadas Quantity (arriba)']  = int(n_qty_sup)
stats_cleaning['Filas capadas Quantity (abajo)']   = int(n_qty_inf)
stats_cleaning['Filas capadas UnitPrice (arriba)'] = int(n_price_sup)

print(f"── 3.5 Capping de outliers (winsorización al p99) ──")
print(f"  Umbral Quantity  (p99 positivos): {cap_qty:.1f} uds  → clip [{-cap_qty:.1f}, {cap_qty:.1f}]")
print(f"  Umbral UnitPrice (p99 positivos): £{cap_price:.2f}  → clip [-, {cap_price:.2f}]")
print()
print(f"  Filas Quantity  > +umbral (capadas arriba): {n_qty_sup:,}")
print(f"  Filas Quantity  < -umbral (capadas abajo) : {n_qty_inf:,}")
print(f"  Filas UnitPrice > umbral  (capadas arriba): {n_price_sup:,}")
print()
print(f"  Filas totales (sin cambio)    : {len(df_working):,}")
print(f"  Quantity  rango tras capping  : [{df_working['Quantity'].min():.1f}, {df_working['Quantity'].max():.1f}]")
print(f"  UnitPrice máxima tras capping : £{df_working['UnitPrice'].max():.2f}")


── 3.5 Capping de outliers (winsorización al p99) ──
  Umbral Quantity  (p99 positivos): 100.0 uds  → clip [-100.0, 100.0]
  Umbral UnitPrice (p99 positivos): £16.63  → clip [-, 16.63]

  Filas Quantity  > +umbral (capadas arriba): 4,886
  Filas Quantity  < -umbral (capadas abajo) : 155
  Filas UnitPrice > umbral  (capadas arriba): 4,889

  Filas totales (sin cambio)    : 531,744
  Quantity  rango tras capping  : [-100.0, 100.0]
  UnitPrice máxima tras capping : £16.63


### 3.5b Eliminar filas con UnitPrice = 0

Tras el capping pueden quedar filas con `UnitPrice = 0` que no son cancelaciones ni ajustes capturables — son registros sin valor económico. Se eliminan.


In [19]:
# 3.5b — Eliminar filas con UnitPrice = 0
antes = len(df_working)
df_working = df_working[df_working['UnitPrice'] > 0].reset_index(drop=True)
eliminadas_price0 = antes - len(df_working)
stats_cleaning['UnitPrice = 0 eliminados'] = eliminadas_price0

print(f"── 3.5b Eliminar filas con UnitPrice = 0 ──")
print(f"  Filas antes     : {antes:,}")
print(f"  Filas eliminadas: {eliminadas_price0:,}")
print(f"  Filas después   : {len(df_working):,}")
print(f"  Verificación    — UnitPrice = 0 restantes: {(df_working['UnitPrice'] == 0).sum()}")


── 3.5b Eliminar filas con UnitPrice = 0 ──
  Filas antes     : 531,744
  Filas eliminadas: 572
  Filas después   : 531,172
  Verificación    — UnitPrice = 0 restantes: 0


### 3.5c Eliminar filas con Quantity = 0

Filas con `Quantity = 0` no representan ninguna transacción real (ni venta ni devolución) y su `TotalPrice` es siempre 0. Se eliminan.


In [20]:
# 3.5c — Eliminar filas con Quantity = 0
antes = len(df_working)
df_working = df_working[df_working['Quantity'] != 0].reset_index(drop=True)
eliminadas_qty0 = antes - len(df_working)
stats_cleaning['Quantity = 0 eliminados'] = eliminadas_qty0

print(f"── 3.5c Eliminar filas con Quantity = 0 ──")
print(f"  Filas antes     : {antes:,}")
print(f"  Filas eliminadas: {eliminadas_qty0:,}")
print(f"  Filas después   : {len(df_working):,}")
print(f"  Verificación    — Quantity = 0 restantes: {(df_working['Quantity'] == 0).sum()}")


── 3.5c Eliminar filas con Quantity = 0 ──
  Filas antes     : 531,172
  Filas eliminadas: 0
  Filas después   : 531,172
  Verificación    — Quantity = 0 restantes: 0


In [ ]:
# [DOC] Comenzamos a verificar la integridad de los datos una vez realizada la limpieza, primero generamos un heatmap

plt.figure(figsize=(10, 3))
sns.heatmap(df_working.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title('Verificación Final: Mapa de Presencia de Datos')
plt.show()

In [ ]:
# [DOC] Ahora generamos comparativas entre los datos brutos y los datos limpios de los campos de cantidad y precio unitario

fig, ax = plt.subplots(2, 2, figsize=(16, 10))

# [DOC] Histograma para Quantity
sns.histplot(df_raw['Quantity'], bins=50, ax=ax[0,0], color='lightgrey', kde=False)
ax[0,0].set_title('Quantity: Distribución Original')
ax[0,0].set_yscale('log')

sns.histplot(df_working['Quantity'], bins=50, ax=ax[0,1], color='skyblue', kde=True)
ax[0,1].set_title('Quantity: Distribución Saneada')

# [DOC] Histograma para UnitPrice
sns.histplot(df_raw['UnitPrice'], bins=50, ax=ax[1,0], color='lightgrey', kde=False)
ax[1,0].set_title('UnitPrice: Distribución Original')
ax[1,0].set_yscale('log')

sns.histplot(df_working['UnitPrice'], bins=50, ax=ax[1,1], color='salmon', kde=True)
ax[1,1].set_title('UnitPrice: Distribución Saneada')

plt.tight_layout()
plt.show()

In [ ]:
# [DOC] Esta última visualización es un balance de datos entre el dataset original y el dataset saneado

labels = ['Datos Conservados', 'Datos Eliminados (Ruido)']
sizes = [len(df_working), len(df_raw) - len(df_working)]

plt.figure(figsize=(7, 7))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140, colors=['#66b3ff','#ff9999'], explode=(0.1, 0))
plt.title('Balance Final del Proceso de Saneamiento')
plt.show()

In [ ]:
# [DOC] Por último, exportamos los datos saneados y limpios

output_dir = '../../data/interim'
output_path = os.path.join(output_dir, 'data_sanitized.csv')

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"[INFO] Carpeta creada: {output_dir}")

df_working.to_csv(output_path, index=False)

print(f"[SUCCESS] Proceso finalizado. Dataset guardado en: {output_path}")
print(f"[INFO] Resumen técnico: {stats_cleaning}")